## Лабораторная работа 13.

1. С помощью библиотеки torch создать модель с прямым проходом, состоящую из 3 слоёв* с функциями активации: ReLu, ReLu, Softmax. 
2. Обучить нейросеть распознавать рукописные цифры на датасете MNIST (28х28 px). 
* Два первых слоя могут быть полносвязные или свёрточные на ваш выбор. Последний слой - это FC слой с 10 нейронами.

In [4]:
import torch 
import torch.nn as nn
import torch.nn.functional as F

In [5]:
# input 28*28*1 вроде не rgb
class MnistNet(nn.Module):
    def __init__(self):
        super(MnistNet, self).__init__() #почему-то без него не работает
        self.conv1 = nn.Conv2d(in_channels = 1, out_channels=32, kernel_size=3, padding=1)#28*28 - 28*28
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)  # 28*28 - 28*28
        self.pool = nn.MaxPool2d(2, 2)  # 28-14, 14-7

        self.fc1 = nn.Linear(64*7*7, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = self.pool(x)
        x = F.relu(self.conv2(x))
        x = self.pool(x)
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        x = F.log_softmax(x, dim=1)  # ← это и есть "Softmax" для NLLLoss
        return x

In [6]:
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

transform = transforms.ToTensor()
trainset = torchvision.datasets.MNIST(
    root="./data", train=True, download=True, transform=transform
)
trainloader = DataLoader(trainset, batch_size=64, shuffle=True)

model = MnistNet()
criterion = nn.NLLLoss()  
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

model.train()
for epoch in range(5):
    running_loss = 0.0
    for images, labels in trainloader:
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    print(f"Epoch {epoch+1}, Loss: {running_loss/len(trainloader):.4f}")


Epoch 1, Loss: 0.1753
Epoch 2, Loss: 0.0496
Epoch 3, Loss: 0.0327
Epoch 4, Loss: 0.0250
Epoch 5, Loss: 0.0186
